# Every delivery time of three apps as a dot

**Dot Plot** · a worked example from the StatNex teaching guide

**The story.** The table holds the delivery times in minutes of 50 orders from each of three food-delivery apps: ZipEats, FeastNow and UrbanBite. A dot plot puts every single order on the chart, which is useful when the audience distrusts summaries.

**The question.** What do the raw delivery times of each app look like when nothing is hidden?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import matplotlib
from IPython import get_ipython
from IPython.display import Image, display

matplotlib.use("Agg")
shown = set()


def show_new_figures(*_):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path, path.stat().st_mtime_ns) not in shown:
            shown.add((path, path.stat().st_mtime_ns))
            display(Image(filename=str(path)))


events = get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `01_basic/dot_plot.csv` (50 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,ZipEats,FeastNow,UrbanBite
1.0,16.0,32.0,36.0
2.0,24.0,26.0,41.0
3.0,28.0,31.0,38.0
4.0,19.0,33.0,48.0
5.0,29.0,25.0,33.0
6.0,26.0,37.0,42.0
7.0,22.0,30.0,42.0
8.0,30.0,28.0,42.0
9.0,22.0,29.0,34.0
10.0,26.0,35.0,46.0
11.0,28.0,33.0,38.0
12.0,17.0,27.0,53.0
13.0,29.0,34.0,41.0
14.0,26.0,26.0,45.0
15.0,23.0,23.0,35.0
16.0,23.0,30.0,46.0
17.0,23.0,30.0,47.0
18.0,22.0,33.0,33.0
19.0,26.0,35.0,44.0
20.0,12.0,29.0,31.0
21.0,26.0,29.0,32.0
22.0,14.0,42.0,44.0
23.0,29.0,33.0,42.0
24.0,20.0,32.0,39.0
25.0,23.0,32.0,41.0
26.0,20.0,41.0,38.0
27.0,27.0,29.0,46.0
28.0,23.0,36.0,38.0
29.0,28.0,34.0,36.0
30.0,19.0,33.0,31.0
31.0,27.0,30.0,36.0
32.0,27.0,32.0,38.0
33.0,29.0,38.0,32.0
34.0,24.0,35.0,46.0
35.0,26.0,41.0,45.0
36.0,18.0,31.0,38.0
37.0,25.0,37.0,44.0
38.0,58.0,37.0,34.0
39.0,17.0,33.0,32.0
40.0,31.0,31.0,29.0
41.0,27.0,40.0,45.0
42.0,18.0,39.0,40.0
43.0,24.0,35.0,37.0
44.0,29.0,37.0,42.0
45.0,25.0,35.0,43.0
46.0,24.0,35.0,36.0
47.0,25.0,31.0,39.0
48.0,23.0,33.0,40.0
49.0,20.0,37.0,46.0
50.0,26.0,33.0,44.0

## Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
# One column per variable. Blank and text cells are left out, so columns may differ
# in length.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["ZipEats", "FeastNow", "UrbanBite"]
series = {c: pd.to_numeric(data[c], errors="coerce").dropna().to_numpy(float) for c in columns}

## Descriptive statistics

In [ ]:
# Variance and SD divide by n - 1. Quartiles interpolate linearly between the sorted
# values (NumPy's default, Excel's QUARTILE.INC).
def describe(x):
    q1, median, q3 = np.percentile(x, [25, 50, 75])
    return {"n": x.size, "mean": np.mean(x), "variance": np.var(x, ddof=1),
            "sd": np.std(x, ddof=1), "min": x.min(), "q1": q1, "median": median,
            "q3": q3, "max": x.max(), "iqr": q3 - q1}


summary = pd.DataFrame({name: describe(x) for name, x in series.items()})
print(summary)

## Dot plot

In [ ]:
# One dot per observation, one row per variable: nothing is summarised away, so read
# it for clusters, gaps and lone values far from the rest. A value that repeats is
# stacked upwards rather than jittered, so the picture is the same every time.
# Matplotlib has no dot-plot function, so the stacking is written out. A dot plot
# stays readable for dozens of values per row; for thousands, a histogram or box plot
# is easier to read.
Path("figures").mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(7, 1 + 1.2 * len(series)))
for row, (name, x) in enumerate(series.items()):
    seen = {}
    heights = []
    for value in x:
        heights.append(row + 0.06 * seen.get(value, 0))
        seen[value] = seen.get(value, 0) + 1
    ax.scatter(x, heights, s=14)
ax.set_yticks(range(len(series)), list(series))
ax.set_xlabel("Value")
fig.tight_layout()
fig.savefig("figures/dot_plot.png", dpi=150)
plt.close(fig)

## What the result says

A dot plot draws one dot for every order, so nothing is hidden. Each app forms one main cluster. The clusters move later from ZipEats to FeastNow to UrbanBite: the means (averages) are 24.46, 32.94 and 39.76 minutes.

Only ZipEats has a dot far from its cluster, at 58 minutes, the maximum. That one order raises ZipEats' standard deviation, the typical distance from the mean, to 6.49 minutes. FeastNow has 4.23 and UrbanBite 5.34.

So the table makes ZipEats look the least consistent, but the dots show that this comes from one order. For most customers ZipEats is as steady as FeastNow: both have an interquartile range, the spread of the middle half, of 5 minutes.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.